# Dataplex Catalog + Self-Discovering Agent (March 2026 Suite)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_dataplex_catalog_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_dataplex_catalog_demo.ipynb)

This notebook shows how an ADK agent can search the Dataplex Catalog to find tables it doesn't know about, read their schemas, and query them — all without being told where the data lives.

## Use Case
A data scientist doesn't know which table has customer churn data. Instead of digging through docs, they ask the agent:
1.  **Find it**: Search Dataplex Catalog for 'churn' across the project.
2.  **Understand it**: Retrieve the table schema.
3.  **Query it**: Run a query to answer the question.

### Release Notes
- [ADK v1.27.0](https://github.com/google/adk-python/releases/tag/v1.27.0) — Dataplex Catalog search tool added to BigQuery ADK
- [ADK v1.27.2](https://github.com/google/adk-python/releases/tag/v1.27.2) — Fix: valid Dataplex OAuth scope for BigQueryToolset

### Requirements
- BigQuery API and Dataplex API enabled.
- `google-adk >= 1.28.0` installed.
- Data Catalog / Dataplex Viewer permissions.

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" google-genai google-cloud-bigquery nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'us-central1' # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. [MANDATORY] Project Configuration

Enable the Dataplex and BigQuery APIs.

In [ ]:
# Enable services
!gcloud services enable bigquery.googleapis.com dataplex.googleapis.com --project={project_id}
print("Success: APIs enabled.")

### 3. [PREREQUISITES] Infrastructure Setup

Create a 'hidden' table that the agent will discover via Dataplex.

In [ ]:
from google.cloud import bigquery

def setup_discoverable_data():
    client = bigquery.Client(project=project_id, location=location)
    dataset_id = f"{project_id}.enterprise_data"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = location
    client.create_dataset(dataset, exists_ok=True)

    table_id = f"{dataset_id}.customer_churn_2026"
    schema = [
        bigquery.SchemaField("customer_id", "STRING", description="Unique ID for the customer"),
        bigquery.SchemaField("churn_score", "FLOAT", description="Probability of churn (0.0 to 1.0)"),
        bigquery.SchemaField("last_active_date", "DATE"),
    ]
    # Recreate table to avoid duplicate data on re-run
    client.delete_table(table_id, not_found_ok=True)
    table = bigquery.Table(table_id, schema=schema)
    table.description = "Master table for 2026 customer churn analysis."
    client.create_table(table)
    
    client.insert_rows_json(table_id, [
        {"customer_id": "C_001", "churn_score": 0.85, "last_active_date": "2026-03-01"}
    ])
    print(f"Target table created at {table_id}. Ready for discovery.")

setup_discoverable_data()

### 4. Core Feature: Self-Discovery via BigQuery Toolset

The ADK BigQuery Toolset lets an agent search for tables, read their schemas, and run queries — so it can explore data on its own.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.adk.tools.bigquery import BigQueryToolset
from google.genai import types

# 1. Initialize the BQ Toolset (captures GOOGLE_CLOUD_LOCATION from env — regional)
bq_toolset = BigQueryToolset()

# 2. Gemini 3.1 Pro Preview is only available in the 'global' location.
#    Switch env var AFTER BigQueryToolset init so BQ keeps using regional location.
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# 3. Define the Self-Discovering Agent
agent = Agent(
    model="gemini-3.1-pro-preview",
    name="DiscoveryAssistant",
    instruction="""
    You are a helpful data assistant. 
    If you don't know where a specific data table is, use your BigQuery tools to search for it. 
    Once found, retrieve its schema and answer the user's question with a query.
    """,
    tools=[bq_toolset]
)

# 4. Initialize Runner
runner = Runner(
    agent=agent,
    session_service=InMemorySessionService(),
    app_name="dataplex_discovery_demo",
    auto_create_session=True
)

async def run_discovery_demo():
    user_query = "Find the churn score for customer C_001. I don't know which table has this data."
    print(f"User: {user_query}\n")
    
    message = types.Content(parts=[types.Part(text=user_query)], role='user')
    async for event in runner.run_async(
        user_id="partner_user",
        session_id="march_session",
        new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent: {part.text}")
                if part.function_call:
                    print(f"[SYSTEM]: Calling tool '{part.function_call.name}'")

await run_discovery_demo()

### 5. Things to remember or know
- **Self-discovery**: Agents aren't limited to pre-configured schemas. They can search the entire Dataplex Catalog to find relevant tables on their own.
- **Schema-aware SQL**: The Dataplex tool returns table locations and column descriptions, so the agent writes accurate queries without guessing.
- **IAM-respecting**: Discovery follows the user's Dataplex permissions — the agent can only find data the user is authorized to see.
- **Runner pattern**: All March 2026 demos use the `Runner` for automatic session management and event streaming.
- **Availability**: Part of ADK v1.27, released March 11, 2026.